# Deep Retention - Ejecucion en Google Colab

Reproduce el proyecto de punta a punta: instala dependencias, corre los tests, entrena la red,
muestra metricas, figuras y resumen financiero, y levanta la aplicacion Streamlit con un link publico temporal.

`Entorno de ejecucion > Ejecutar todo`. CPU es suficiente (el entrenamiento tarda ~1 minuto).

In [ ]:
# Repositorio y rama a ejecutar
REPO_URL = 'https://github.com/jose30a/proyecto-1-ia.git'
BRANCH = 'main'

!rm -rf /content/proyecto-1-ia
!git clone -q --branch {BRANCH} {REPO_URL} /content/proyecto-1-ia
%cd /content/proyecto-1-ia
!git log --oneline -1
!pip install -q -r requirements.txt

## 1. Tests (unitarios + integracion)

In [ ]:
!python -m pytest -q tests

## 2. Entrenamiento (semilla fija: los resultados son reproducibles)

In [ ]:
!python train.py

## 3. Metricas, calibracion y explicabilidad

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
m = json.load(open('models/test_metrics.json'))
cols = ['roc_auc', 'pr_auc', 'brier_score', 'ece', 'mean_predicted', 'observed_rate']
display(pd.DataFrame({'Sin calibrar': {k: m['uncalibrated'][k] for k in cols},
                      'Calibrada (Platt)': {k: m[k] for k in cols}}).round(4))
print('Mejor epoca:', m['best_epoch'], '| Epocas ejecutadas:', m['epochs_run'])
for fig in ('training_curve', 'calibration_curve', 'shap_importance'):
    display(Image(f'docs/figures/{fig}.png', width=520))

## 4. Resumen financiero (cartera completa y holdout de Test)

In [ ]:
fs = json.load(open('models/financial_summary.json'))
display(pd.DataFrame({'Cartera completa': fs['portfolio'], 'Holdout Test': fs['test_holdout']}))
sc = pd.read_csv('data/portfolio_scored.csv')
display(sc.groupby(['segment', 'decision', 'incentive_name']).agg(
    clientes=('p_churn', 'size'), p_media=('p_churn', 'mean'), saldo_medio=('Balance', 'mean'),
    costo=('incentive_cost', 'sum'), ganancia_neta=('expected_net_gain', 'sum'), fuga_real=('Exited', 'mean')).round(2))

## 5. Aplicacion Streamlit
Abre el link `https://...trycloudflare.com` que imprime la celda (la primera carga tarda unos segundos).
Para detenerla: `Entorno de ejecucion > Interrumpir`.

In [ ]:
import subprocess, re, time
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
app = subprocess.Popen(['streamlit', 'run', 'app/app.py', '--server.port', '8501', '--server.headless', 'true'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    found = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if found:
        print('App disponible en:', found.group(0))
        break